In [6]:
import numpy as np
from pathlib import Path
import footsiesgym
from footsiesgym.footsies.game import constants

# Configuration: use the existing macOS game server on port 50051.
PLATFORM = "mac"
LAUNCH_BINARIES = False
PORT = 50051
TRAIN_EPISODES = 300
MAX_STEPS = 600
EVAL_EVERY = 50
QUICK_EVAL_EPISODES = 8
FINAL_EVAL_EPISODES = 30
ALPHA, GAMMA = 0.20, 0.98
EPS_START, EPS_END = 0.40, 0.05
GUARD_REWARD = 0.10
CHECKPOINT_DIR = Path("footsies_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# 4 distance bins x 3 opponent phases x 6 actions = 72 Q-values.
DISTANCE_BINS = [0.07, 0.15, 0.30]
NUM_ACTIONS = 6
Q = np.zeros((12, NUM_ACTIONS), dtype=np.float64)

# The opponent's public state occupies the last 37 observation entries.
ACTION_NAMES = list(constants.FOOTSIES_ACTION_IDS)
ATTACK_NAMES = ("N_ATTACK", "B_ATTACK", "N_SPECIAL", "B_SPECIAL")
ATTACK_INDICES = {ACTION_NAMES.index(name) for name in ATTACK_NAMES}


def distance_bin(obs):
    return int(np.digitize(float(obs[0]), DISTANCE_BINS))


def opponent_phase(obs):
    opp = np.asarray(obs)[-37:]
    if opp[32] > 0.5:
        return 2  # Opponent is stunned.
    action_index = int(np.argmax(opp[8:25]))
    if action_index not in ATTACK_INDICES:
        return 0  # No current attack animation.
    progress = float(opp[25])
    duration = max(float(opp[26]), 1e-6)
    return 1 if progress / duration < 0.65 else 2  # Late attack is a recovery proxy.


def encode_state(obs):
    return distance_bin(obs) * 3 + opponent_phase(obs)


def guard_health(obs, opponent=False):
    block = np.asarray(obs)[-37:] if opponent else np.asarray(obs)[1:]
    return int(np.argmax(block[4:8]))


def opponent_action(obs, rng):
    d = float(obs[0])
    if d > 0.30:
        actions, probs = [2, 1, 5], [0.80, 0.10, 0.10]
    elif d > 0.15:
        actions, probs = [2, 5, 3, 1], [0.50, 0.25, 0.15, 0.10]
    elif d > 0.07:
        actions, probs = [3, 5, 1, 0], [0.45, 0.20, 0.20, 0.15]
    else:
        actions, probs = [3, 4, 1, 0], [0.55, 0.25, 0.15, 0.05]
    return int(rng.choice(actions, p=probs))


def choose_action(obs, mode, epsilon, rng):
    if mode == "random":
        return int(rng.integers(NUM_ACTIONS))
    if mode == "hardcoded":
        return 2 if float(obs[0]) > 0.20 else 3
    if mode != "q":
        raise ValueError(f"Unknown mode: {mode}")
    
    if rng.random() < epsilon:
        return int(rng.integers(NUM_ACTIONS))
    
    values = Q[encode_state(obs)]
    return int(rng.choice(np.flatnonzero(values == values.max())))


def run_episode(env, mode="q", seed=0, train=False, epsilon=0.0):
    agent_rng = np.random.default_rng(seed + 100)
    opponent_rng = np.random.default_rng(seed + 200)
    obs, _ = env.reset(seed=seed)
    total_game_reward, outcome = 0.0, 0

    for step in range(MAX_STEPS):
        old_obs = obs["p1"]
        state = encode_state(old_obs)
        action = choose_action(old_obs, mode, epsilon, agent_rng)
        opp_action = opponent_action(obs["p2"], opponent_rng)
        next_obs, rewards, terminated, truncated, _ = env.step(
            {"p1": action, "p2": opp_action}
        )
        r = float(rewards["p1"])
        done = bool(terminated["p1"] or truncated["p1"])
        total_game_reward += r

        if train and mode == "q":
            # Use small guard-hit bonuses only for learning, not for evaluation.
            self_loss = guard_health(old_obs) - guard_health(next_obs["p1"])
            opp_loss = guard_health(old_obs, opponent=True) - guard_health(
                next_obs["p1"], opponent=True
            )
            shaped_reward = r + GUARD_REWARD * (opp_loss - self_loss)
            target = shaped_reward
            if not done:
                target += GAMMA * np.max(Q[encode_state(next_obs["p1"])])
            Q[state, action] += ALPHA * (target - Q[state, action])

        if done:
            if terminated["p1"]:
                outcome = int(np.sign(r))
            break
        obs = next_obs

    return total_game_reward, outcome, step + 1


def evaluate(env, mode="q", episodes=30, start_seed=10000):
    before = Q.copy()
    results = [
        run_episode(env, mode, seed=start_seed + i, train=False, epsilon=0.0)
        for i in range(episodes)
    ]
    assert np.array_equal(Q, before), "Evaluation unexpectedly updated Q!"
    returns, outcomes, lengths = np.asarray(results).T
    
    return {
        "return": float(returns.mean()),
        "win_rate": float(np.mean(outcomes == 1)),
        "loss_rate": float(np.mean(outcomes == -1)),
        "avg_steps": float(lengths.mean()),
    }


env = footsiesgym.make(
    config={
        "port": PORT,
        "headless": True,
        "max_t": MAX_STEPS,
        "action_delay": 0,
    },
    platform=PLATFORM,
    launch_binaries=LAUNCH_BINARIES,
)

try:
    assert env.action_space("p1").n == NUM_ACTIONS
    
    obs, _ = env.reset()
    
    assert len(obs["p1"]) == 88, "Unexpected observation layout."
    assert encode_state(obs["p1"]) in range(12)
    
    print("Q-table shape:", Q.shape)

    best_ep, best_score = None, -float("inf")
    
    for ep in range(1, TRAIN_EPISODES + 1):
        epsilon = EPS_START + (EPS_END - EPS_START) * (ep - 1) / max(1, TRAIN_EPISODES - 1)
        run_episode(env, mode="q", seed=ep, train=True, epsilon=epsilon)
        
        if ep % EVAL_EVERY == 0:
            np.save(CHECKPOINT_DIR / f"Q_ep{ep}.npy", Q)
            stats = evaluate(env, "q", QUICK_EVAL_EPISODES)
            
            if stats["return"] > best_score:
                best_ep, best_score = ep, stats["return"]
                
            print(f"Episode {ep:3d} | epsilon={epsilon:.2f} | eval return={stats['return']:.3f}")

    # Restore the best saved policy rather than automatically using episode 300.
    Q[:] = np.load(CHECKPOINT_DIR / f"Q_ep{best_ep}.npy")
    np.save(CHECKPOINT_DIR / "Q_best.npy", Q)
    
    print(f"\nBest checkpoint: episode {best_ep} (quick-eval return={best_score:.3f})\n")
    
    for mode in ("random", "hardcoded", "q"):
        stats = evaluate(env, mode, FINAL_EVAL_EPISODES, start_seed=20000)
        print(f"{mode:9s} | return={stats['return']:+.3f} | "
              f"win={stats['win_rate']:.1%} | loss={stats['loss_rate']:.1%}")
    print("Saved checkpoint:", CHECKPOINT_DIR / "Q_best.npy")
finally:
    env.close()


Q-table shape: (12, 6)


Error calling StepNFrames with exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "Stream removed (Socket closed)"
	debug_error_string = "UNAVAILABLE:Stream removed (Socket closed)"
>


_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "Stream removed (Socket closed)"
	debug_error_string = "UNAVAILABLE:Stream removed (Socket closed)"
>